In [ ]:
import ee
import geemap
import os
from pyproj import Transformer

In [ ]:


# initialization
ee.Initialize(project="project-8e6c1255-803c-4395-88f")


# swiss grid coordinates -> lon lat
transformer = Transformer.from_crs("EPSG:21781", "EPSG:4326")

def get_glacier_view(coordx, coordy, name):
    # convert coordinates -> lon/lat
    lat, lon = transformer.transform(coordx, coordy)
    roi = ee.Geometry.Point([lon, lat]).buffer(3000).bounds()

    # least cloudy image from late summer
    image = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .filterDate("2024-08-01", "2024-09-30")
        .sort("CLOUDY_PIXEL_PERCENTAGE")
        .first()
        .divide(10000)
    )

    # snow/ice detection 
    ndsi = image.normalizedDifference(["B3", "B11"]).rename("NDSI")
    nir = image.select("B8")
    ice_mask = ndsi.gt(0.4).And(nir.gt(0.11)).rename("ice_mask")

   
    vis_image = image.select(['B4', 'B3', 'B2']).visualize(min=0, max=0.3)
    
    output_dir = '/Users/maraeckart/dev/hslu/fs26/DSPRO/I.BA_DSPRO2/processed_glacier_image/'
    image_path = os.path.join(output_dir, f"{name}_2024.tif")
    
    try:
        print(f"Attempting to save image to: {image_path}...")
        geemap.ee_export_image(
            vis_image, 
            filename=image_path, 
            scale=30,     
            region=roi, 
            file_per_band=False
        )
        print("Download successful.")
    except Exception as e:
        print(f"Could not download image for {name}: {e}")
        image_path = None

    # DEM 
    dem = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").mosaic()
    ice_elevations = dem.updateMask(ice_mask)

    # SLA
    sla_stats = ice_elevations.reduceRegion(
        reducer=ee.Reducer.percentile([5]),
        geometry=roi,
        scale=30,
        maxPixels=1e9,
    )
    sla_value = sla_stats.values().get(0).getInfo() 
    print(f"Snow Line Altitude (SLA): {sla_value} m")

    # area in km^2
    area_image = ice_mask.multiply(ee.Image.pixelArea())
    area_stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
    )
    area_m2 = area_stats.values().get(0) # Robust way to get the value
    glacier_area_km2 = ee.Number(area_m2).divide(1e6).getInfo()
    print(f"Detected glacier area: {glacier_area_km2:.2f} km^2")

    # map
    m = geemap.Map()
    m.centerObject(roi, 13)
    m.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, f"real {name}")
    m.addLayer(ice_mask.updateMask(ice_mask), {"palette": ["#00FFFF"]}, "mask Ice/Snow")
    
    return m, image_path

a_x, a_y = 640640, 137630
m, saved_path = get_glacier_view(a_x, a_y, "aletsch")
m

In [ ]:
import geopandas as gpd

glamos_path = "/Users/maraeckart/dev/hslu/fs26/DSPRO/I.BA_DSPRO2/processed_glamos_data/glacier_geometry_2013-2018 (1).parquet"
debris_path = "/Users/maraeckart/dev/hslu/fs26/DSPRO/I.BA_DSPRO2/processed_glamos_data/debris_geometry_2011-2017 (1).parquet"

gdf = gpd.read_parquet(glamos_path)
gdf_2 = gpd.read_parquet(debris_path)


print(gdf.columns)
print(gdf_2.columns)

print(gdf['geometry'].head())

print(gdf.crs)
gdf.plot()
